# Content-Based Filtering (Anime)\n
\n
This notebook builds a **simple baseline** content-based recommender with:\n
- Synopsis embeddings using `all-mpnet-base-v2`\n
- Genre similarity via Jaccard\n
- Exact-match features for `Type`, `Studios`, and `Source`\n
\n
No custom weighting is used; all components contribute equally.

## 1) Imports

In [ ]:
from __future__ import annotations

import ast
from pathlib import Path

import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer

ModuleNotFoundError: No module named 'sentence_transformers'

## 2) Load Data

In [ ]:
DATA_PATH = Path("data/processed/anime.parquet")

if not DATA_PATH.exists():
    raise FileNotFoundError(f"Could not find dataset at {DATA_PATH}")

df = pd.read_parquet(DATA_PATH)
print(f"Loaded rows: {len(df):,}")
print(f"Columns: {list(df.columns)}")

## 3) Resolve Required Columns

In [ ]:
def resolve_column(frame: pd.DataFrame, preferred: list[str]) -> str:
    for col in preferred:
        if col in frame.columns:
            return col
    raise ValueError(f"None of {preferred} found. Available: {list(frame.columns)}")

title_col = resolve_column(df, ["Name", "name", "Title", "title"])
synopsis_col = resolve_column(df, ["Synopsis", "Synposis", "description", "Description"])
genres_col = resolve_column(df, ["Genres", "genres", "Genre", "genre"])
type_col = resolve_column(df, ["Type", "type"])
studios_col = resolve_column(df, ["Studios", "studios", "Studio", "studio"])
source_col = resolve_column(df, ["Source", "source"])
score_col = resolve_column(df, ["Score", "score", "Rating", "rating"])
votes_col = resolve_column(df, ["Scored By", "scored_by", "Votes", "votes"])

print("Resolved columns:")
print("title:", title_col)
print("synopsis:", synopsis_col)
print("genres:", genres_col)
print("type:", type_col)
print("studios:", studios_col)
print("source:", source_col)
print("score:", score_col)
print("votes:", votes_col)

## 4) Basic Preparation

In [ ]:
def parse_genre_set(value) -> set[str]:
    if pd.isna(value):
        return set()

    text = str(value).strip()
    if not text:
        return set()
    if text.startswith("[") and text.endswith("]"):
        try:
            parsed = ast.literal_eval(text)
            if isinstance(parsed, list):
                return {str(item).strip().lower() for item in parsed if str(item).strip()}
        except (ValueError, SyntaxError):
            pass

    parts = [part.strip().lower() for part in text.split(",")]
    return {part for part in parts if part}

work_df = df.copy()
work_df[synopsis_col] = work_df[synopsis_col].fillna("").astype(str)
work_df[title_col] = work_df[title_col].fillna("").astype(str)
work_df[type_col] = work_df[type_col].fillna("").astype(str).str.strip().str.lower()
work_df[studios_col] = work_df[studios_col].fillna("").astype(str).str.strip().str.lower()
work_df[source_col] = work_df[source_col].fillna("").astype(str).str.strip().str.lower()
work_df[score_col] = pd.to_numeric(work_df[score_col], errors="coerce")
work_df[votes_col] = pd.to_numeric(work_df[votes_col], errors="coerce").fillna(0.0)
work_df["_genre_set"] = work_df[genres_col].apply(parse_genre_set)

work_df = work_df[work_df[title_col].str.strip() != ""].reset_index(drop=True)
print(f"Usable rows: {len(work_df):,}")
print(f"Rows with rating: {work_df[score_col].notna().sum():,}")

## 5) Build & Store Synopsis Embeddings (`all-mpnet-base-v2`)

In [ ]:
MODEL_NAME = "sentence-transformers/all-mpnet-base-v2"
model = SentenceTransformer(MODEL_NAME)

synopsis_texts = work_df[synopsis_col].tolist()
embeddings = model.encode(
    synopsis_texts,
    batch_size=32,
    show_progress_bar=True,
    normalize_embeddings=True,
).astype(np.float32)

print("Embedding matrix shape:", embeddings.shape)
print("Embedding dtype:", embeddings.dtype)
print("Note: cosine similarity will be computed on-request via matrix multiplication.")

## 6) Similarity Components

In [ ]:
def jaccard_similarity(a: set[str], b: set[str]) -> float:
    if not a and not b:
        return 0.0
    union = a.union(b)
    if not union:
        return 0.0
    return len(a.intersection(b)) / len(union)

def bayesian_weighted_rating(R: float, v: float, C: float, m: float) -> float:
    denominator = v + m
    if denominator <= 0:
        return C
    return (v / denominator) * R + (m / denominator) * C

genre_sets = work_df["_genre_set"].tolist()
types = work_df[type_col].tolist()
studios = work_df[studios_col].tolist()
sources = work_df[source_col].tolist()

ratings = work_df[score_col].fillna(work_df[score_col].mean()).astype(float).to_numpy()
votes = work_df[votes_col].fillna(0.0).astype(float).to_numpy()

C = float(np.nanmean(ratings))
m = float(np.percentile(votes, 75))

bayesian_scores = np.array([bayesian_weighted_rating(R=ratings[i], v=votes[i], C=C, m=m) for i in range(len(work_df))])

bayes_min = float(np.min(bayesian_scores))
bayes_max = float(np.max(bayesian_scores))
if bayes_max > bayes_min:
    bayesian_scores_norm = (bayesian_scores - bayes_min) / (bayes_max - bayes_min)
else:
    bayesian_scores_norm = np.zeros_like(bayesian_scores)

print(f"Global average rating (C): {C:.4f}")
print(f"Minimum votes threshold (m): {m:.2f}")
print(f"Bayesian score range: [{bayes_min:.4f}, {bayes_max:.4f}]")

## 7) Recommender Utilities (On-Request Similarity + MMR)

In [ ]:
FEATURE_WEIGHTS = {
    "synopsis": 0.45,
    "genre": 0.25,
    "type": 0.10,
    "studio": 0.05,
    "source": 0.05,
    "bayesian": 0.10,
}

weight_sum = sum(FEATURE_WEIGHTS.values())
if weight_sum <= 0:
    raise ValueError("Sum of feature weights must be > 0")

# Cached arrays for faster indexed lookups
titles_arr = work_df[title_col].to_numpy()
types_arr = np.asarray(types, dtype=object)
studios_arr = np.asarray(studios, dtype=object)
sources_arr = np.asarray(sources, dtype=object)
bayesian_scores_norm_arr = np.asarray(bayesian_scores_norm, dtype=np.float32)


def find_query_index(name: str) -> int:
    name_norm = name.strip().lower()
    titles = work_df[title_col].str.lower()

    exact = titles[titles == name_norm]
    if len(exact) > 0:
        return int(exact.index[0])

    contains = titles[titles.str.contains(name_norm, regex=False)]
    if len(contains) > 0:
        return int(contains.index[0])

    raise ValueError(f"Could not find anime with name: {name}")


def apply_mmr(
    candidates_df: pd.DataFrame,
    embeddings_matrix: np.ndarray,
    top_k: int,
    lambda_mmr: float = 0.7,
) -> pd.DataFrame:
    if candidates_df.empty:
        return candidates_df

    if not (0.0 <= lambda_mmr <= 1.0):
        raise ValueError("lambda_mmr must be between 0 and 1")

    pool = candidates_df.copy().reset_index(drop=True)
    selected_rows = []
    selected_indices = []

    while len(selected_rows) < min(top_k, len(pool)):
        best_row_pos = None
        best_mmr_score = -np.inf

        for row_pos in range(len(pool)):
            candidate_index = int(pool.iloc[row_pos]["index"])
            relevance = float(pool.iloc[row_pos]["final_score"])

            if not selected_indices:
                diversity_penalty = 0.0
            else:
                # Embeddings are L2-normalized -> dot product equals cosine similarity
                selected_matrix = embeddings_matrix[selected_indices]
                candidate_vec = embeddings_matrix[candidate_index]
                similarities = selected_matrix @ candidate_vec
                diversity_penalty = float(np.max(similarities))

            mmr_score = lambda_mmr * relevance - (1.0 - lambda_mmr) * diversity_penalty

            if mmr_score > best_mmr_score:
                best_mmr_score = mmr_score
                best_row_pos = row_pos

        chosen = pool.iloc[best_row_pos].copy()
        chosen["mmr_score"] = best_mmr_score
        selected_rows.append(chosen)
        selected_indices.append(int(chosen["index"]))
        pool = pool.drop(index=best_row_pos).reset_index(drop=True)

    return pd.DataFrame(selected_rows).reset_index(drop=True)


def recommend_by_index(
    q_idx: int,
    top_k: int = 10,
    top_k_mmr: int = 40,
    lambda_mmr: float = 0.7,
) -> pd.DataFrame:
    n_total = len(work_df)
    if q_idx < 0 or q_idx >= n_total:
        raise ValueError(f"Invalid q_idx={q_idx}. Must be in [0, {n_total - 1}]")

    if n_total <= 1:
        return pd.DataFrame(columns=["index", "title", "final_score"])
    if top_k_mmr < top_k:
        top_k_mmr = top_k

    n_candidates = min(top_k_mmr, n_total - 1)

    # On-request similarity via matrix multiplication
    query_vec = embeddings[q_idx]
    similarities = embeddings @ query_vec
    similarities[q_idx] = -1.0  # exclude self

    top_indices = np.argpartition(similarities, -n_candidates)[-n_candidates:]
    top_indices = top_indices[np.argsort(similarities[top_indices])[::-1]]

    q_genres = genre_sets[q_idx]
    q_type = types_arr[q_idx]
    q_studio = studios_arr[q_idx]
    q_source = sources_arr[q_idx]

    candidate_indices = top_indices.astype(int)
    synopsis_scores = similarities[candidate_indices].astype(np.float32)
    genre_scores = np.array([jaccard_similarity(q_genres, genre_sets[i]) for i in candidate_indices], dtype=np.float32)

    if q_type:
        type_scores = (types_arr[candidate_indices] == q_type).astype(np.float32)
    else:
        type_scores = np.zeros(len(candidate_indices), dtype=np.float32)

    if q_studio:
        studio_scores = (studios_arr[candidate_indices] == q_studio).astype(np.float32)
    else:
        studio_scores = np.zeros(len(candidate_indices), dtype=np.float32)

    if q_source:
        source_scores = (sources_arr[candidate_indices] == q_source).astype(np.float32)
    else:
        source_scores = np.zeros(len(candidate_indices), dtype=np.float32)

    bayes_scores_norm = bayesian_scores_norm_arr[candidate_indices]

    weighted_total = (
        FEATURE_WEIGHTS["synopsis"] * synopsis_scores
        + FEATURE_WEIGHTS["genre"] * genre_scores
        + FEATURE_WEIGHTS["type"] * type_scores
        + FEATURE_WEIGHTS["studio"] * studio_scores
        + FEATURE_WEIGHTS["source"] * source_scores
        + FEATURE_WEIGHTS["bayesian"] * bayes_scores_norm
    )
    final_scores = weighted_total / weight_sum

    base_ranked = pd.DataFrame(
        {
            "index": candidate_indices,
            "title": titles_arr[candidate_indices],
            "final_score": final_scores,
            "synopsis_score": synopsis_scores,
            "genre_score": genre_scores,
            "type_match": type_scores.astype(int),
            "studio_match": studio_scores.astype(int),
            "source_match": source_scores.astype(int),
            "bayesian_rating_norm": bayes_scores_norm,
        }
    ).sort_values("final_score", ascending=False).reset_index(drop=True)

    mmr_pool = base_ranked.head(top_k_mmr).reset_index(drop=True)
    mmr_ranked = apply_mmr(
        mmr_pool,
        embeddings_matrix=embeddings,
        top_k=top_k,
        lambda_mmr=lambda_mmr,
    )
    return mmr_ranked


def recommend_by_name(
    name: str,
    top_k: int = 10,
    top_k_mmr: int = 40,
    lambda_mmr: float = 0.7,
) -> pd.DataFrame:
    q_idx = find_query_index(name)
    return recommend_by_index(q_idx, top_k=top_k, top_k_mmr=top_k_mmr, lambda_mmr=lambda_mmr)

## 8) Save Artifacts for Web App (Embeddings + Metadata)

In [ ]:
import json

ARTIFACT_DIR = Path("ml/artifacts/content_based_anime_v1")
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

# 1) Metadata table used for rendering and lookup in web app
base_metadata_cols = [title_col, synopsis_col, genres_col, type_col, studios_col, source_col]
optional_cols = [c for c in [score_col, votes_col] if c in work_df.columns]
web_metadata = work_df[base_metadata_cols + optional_cols].copy()
web_metadata = web_metadata.reset_index().rename(columns={"index": "anime_index"})
web_metadata["genre_list"] = work_df["_genre_set"].apply(lambda values: sorted(list(values)))

metadata_path = ARTIFACT_DIR / "anime_metadata.parquet"
web_metadata.to_parquet(metadata_path, index=False)

# 2) Store only embeddings (no dense similarity matrix artifact)
embeddings_path = ARTIFACT_DIR / "synopsis_embeddings.npy"
np.save(embeddings_path, embeddings.astype(np.float32))

# 3) Optional compact feature arrays
feature_arrays_path = ARTIFACT_DIR / "feature_arrays.npz"
np.savez(
    feature_arrays_path,
    bayesian_scores=bayesian_scores,
    bayesian_scores_norm=bayesian_scores_norm,
    ratings=ratings,
    votes=votes,
    anime_index=web_metadata["anime_index"].to_numpy(),
)

# 4) Runtime config used by web app service
config = {
    "artifact_version": "content_based_anime_v1",
    "model_name": MODEL_NAME,
    "created_at": pd.Timestamp.utcnow().isoformat(),
    "columns": {
        "title": title_col,
        "synopsis": synopsis_col,
        "genres": genres_col,
        "type": type_col,
        "studios": studios_col,
        "source": source_col,
        "score": score_col,
        "votes": votes_col,
    },
    "weights": FEATURE_WEIGHTS,
    "bayesian": {
        "C": C,
        "m": m,
        "normalized": True,
    },
    "retrieval": {
        "embedding_file": "synopsis_embeddings.npy",
        "similarity_runtime": "dot_product",
        "top_k_mmr_default": 40,
        "lambda_mmr_default": 0.7,
    },
}

config_path = ARTIFACT_DIR / "config.json"
with open(config_path, "w", encoding="utf-8") as f:
    json.dump(config, f, ensure_ascii=False, indent=2)

print("Saved artifacts:")
print("-", metadata_path)
print("-", embeddings_path)
print("-", feature_arrays_path)
print("-", config_path)

## 9) Simple Demo

In [ ]:
QUERY_NAME = "Death Note"  # change this
TOP_K = 20
TOP_K_MMR = 50
MMR_LAMBDA = 0.6  # 1.0 = only relevance, 0.0 = max diversity

query_index = find_query_index(QUERY_NAME)
print("Query anime:", work_df.iloc[query_index][title_col])

demo_results = recommend_by_name(
    QUERY_NAME,
    top_k=TOP_K,
    top_k_mmr=TOP_K_MMR,
    lambda_mmr=MMR_LAMBDA,
 )
demo_results